In [1]:
import os

In [2]:
%pwd

'/workspaces/MLOps-Laptop-Prediction-System/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/workspaces/MLOps-Laptop-Prediction-System'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
from Laptop_Price_Prediction_System.constants import *
from Laptop_Price_Prediction_System.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(self, config_filepath=Config_Filepath, params_filepath=Params_Filepath, schema_filepath=Schema_Filepath):
        self.config = read_yaml(config_filepath)
        self.schema = read_yaml(schema_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

        return data_ingestion

In [8]:
import os
import urllib.request as request
import zipfile
from Laptop_Price_Prediction_System import logger
from Laptop_Price_Prediction_System.utils.common import get_size

In [ ]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename=self.config.local_data_file
            )
            logger.info(f"Downloaded file: {filename}")
        else:
            logger.info(f"File Already Exists: {get_size(Path(self.config.local_file_data))}")
    
    def extract_zip_file(self):
        unzip_dir = self.config.unzip_dir
        os.makedirs(unzip_dir, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, "r") as zip_ref:
            zip_ref.extractall(unzip_dir)

In [10]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-07-04 10:53:37,198 : INFO : common : Loaded file: config/config.yaml]
[2026-07-04 10:53:37,203 : INFO : common : Loaded file: schema.yaml]
[2026-07-04 10:53:37,209 : INFO : common : Loaded file: params.yaml]
[2026-07-04 10:53:37,211 : INFO : common : Created directory: artifacts]
[2026-07-04 10:53:37,214 : INFO : common : Created directory: artifacts/data_ingestion]
[2026-07-04 10:53:37,681 : INFO : 3298163946 : Downloaded file: artifacts/data_ingestion/laptop_price_data.zip]
